# TSDF PointNet + ModelNet40 Colab Workflow

这个 notebook 采用更稳的流程：

- 从 Drive 中读取 `TSDF.zip`
- 复制到 `/content` 本地磁盘再解压
- 从 Drive 复制 `kaggle.json`
- 下载 ModelNet40 到 `/content`
- 训练和预测尽量直接在 notebook 的 Python cell 中完成，不通过命令行调用脚本
- 训练结果最后再复制回 Drive

开始前请在 `Runtime -> Change runtime type` 中选择 `T4 GPU`。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_ZIP_IN_DRIVE = Path('/content/drive/MyDrive/TSDF.zip')
KAGGLE_JSON_IN_DRIVE = Path('/content/drive/MyDrive/kaggle.json')

LOCAL_ROOT = Path('/content/local_tsdf_workspace')
LOCAL_REPO_ZIP = LOCAL_ROOT / 'TSDF.zip'
REPO_DIR = LOCAL_ROOT / 'TSDF'
MODELNET_ROOT = REPO_DIR / 'data' / 'ModelNet40'
OUTPUT_DIR = REPO_DIR / 'model' / 'pointnet_modelnet40_run'
DRIVE_EXPORT_DIR = Path('/content/drive/MyDrive/TSDF_training_outputs/pointnet_modelnet40_run')

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
print('REPO_ZIP_IN_DRIVE =', REPO_ZIP_IN_DRIVE)
print('KAGGLE_JSON_IN_DRIVE =', KAGGLE_JSON_IN_DRIVE)
print('REPO_DIR =', REPO_DIR)
print('MODELNET_ROOT =', MODELNET_ROOT)
print('OUTPUT_DIR =', OUTPUT_DIR)

In [ ]:
!df -h /content
!df -h /content/drive
!nvidia-smi

## 1. 将仓库 zip 复制到本地磁盘并解压

In [ ]:
import shutil
import zipfile

if not REPO_ZIP_IN_DRIVE.exists():
    raise FileNotFoundError(f'Repo zip not found: {REPO_ZIP_IN_DRIVE}')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
if LOCAL_REPO_ZIP.exists():
    LOCAL_REPO_ZIP.unlink()

extract_root = LOCAL_ROOT / 'repo_extract_tmp'
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)

shutil.copy2(REPO_ZIP_IN_DRIVE, LOCAL_REPO_ZIP)
print('Copied repo zip to', LOCAL_REPO_ZIP)

with zipfile.ZipFile(LOCAL_REPO_ZIP, 'r') as zf:
    zf.extractall(extract_root)

if (extract_root / 'TSDF').exists():
    shutil.move(str(extract_root / 'TSDF'), str(REPO_DIR))
else:
    extracted_dirs = [p for p in extract_root.iterdir() if p.is_dir()]
    if len(extracted_dirs) == 1:
        shutil.move(str(extracted_dirs[0]), str(REPO_DIR))
    else:
        raise RuntimeError('Could not determine extracted repo directory automatically.')

if LOCAL_REPO_ZIP.exists():
    LOCAL_REPO_ZIP.unlink()

print('Repo extracted to', REPO_DIR)
!ls -lah {REPO_DIR}

## 2. 安装依赖

In [ ]:
%cd /content/local_tsdf_workspace/TSDF
!python -V
!pip install -q --upgrade pip
!pip install -q kaggle wandb opencv-python open3d trimesh h5py tqdm pyyaml pillow scipy

In [ ]:
import os
import sys
import torch

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR.parent))

print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 3. 配置 Kaggle API

In [ ]:
import os
import shutil

kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)

if not KAGGLE_JSON_IN_DRIVE.exists():
    raise FileNotFoundError(f'kaggle.json not found: {KAGGLE_JSON_IN_DRIVE}')

shutil.copy2(KAGGLE_JSON_IN_DRIVE, kaggle_dir / 'kaggle.json')
os.chmod(kaggle_dir / 'kaggle.json', 0o600)
print('Kaggle credential installed at', kaggle_dir / 'kaggle.json')

In [ ]:
!kaggle datasets metadata balraj98/modelnet40-princeton-3d-object-dataset -p {MODELNET_ROOT}

## 4. 下载 ModelNet40

如果数据已经存在，则跳过下载。

In [ ]:
import subprocess

train_list = MODELNET_ROOT / 'modelnet40_train.txt'
test_list = MODELNET_ROOT / 'modelnet40_test.txt'

if train_list.exists() and test_list.exists():
    print('ModelNet40 already exists, skip download.')
else:
    MODELNET_ROOT.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable,
        str(REPO_DIR / 'dataset' / 'download_modelnet40_kaggle.py'),
        '--output-dir',
        str(MODELNET_ROOT),
    ]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)

In [ ]:
!find {MODELNET_ROOT} -maxdepth 4 | head -n 120

## 5. 检查数据集类别和 split

In [ ]:
from TSDF.dataset.modelnet40_data import build_modelnet40_splits

labels, train_samples, test_samples = build_modelnet40_splits(MODELNET_ROOT)
print('num_labels =', len(labels))
print('first_10_labels =', labels[:10])
print('train_samples =', len(train_samples))
print('test_samples =', len(test_samples))

## 6. DataLoader 冒烟测试

In [ ]:
from TSDF.dataset.modelnet40_data import get_modelnet40_dataloaders

train_dataset, test_dataset, train_loader, test_loader = get_modelnet40_dataloaders(
    root=MODELNET_ROOT,
    batch_size=4,
    num_points=1024,
    workers=0,
    seed=0,
    sample_method='surface',
)

print('train size =', len(train_dataset))
print('test size =', len(test_dataset))
points, labels_batch = next(iter(train_loader))
print('points shape =', points.shape)
print('labels shape =', labels_batch.shape)

## 7. 训练前单样本耗时测试

如果这里很慢，说明主要瓶颈在 `.off` 读取与采样。

In [ ]:
import time
from TSDF.dataset.modelnet40_data import ModelNet40Dataset

ds = ModelNet40Dataset(
    root=MODELNET_ROOT,
    split='train',
    num_points=1024,
    normalize=True,
    augment=True,
    seed=0,
    sample_method='surface',
)

t0 = time.time()
x, y = ds[0]
t1 = time.time()
print('one sample time =', t1 - t0, 'sec')
print('x shape =', x.shape, 'label =', y)

## 8. 在 notebook 内直接训练 PointNet

这里不通过脚本命令行，而是直接导入模型、dataloader 和训练逻辑。

默认使用：

- `batch_size = 8`
- `workers = 0`
- `epochs = 150`
- 输出目录：`model/pointnet_modelnet40_run`

In [ ]:
import json
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from TSDF.dataset.modelnet40_data import get_modelnet40_dataloaders
from TSDF.detection.pointnet_model import PointNetCls, feature_transform_regularizer
from TSDF.detection.train_pointnet_cls import compute_class_weights, set_seed, evaluate

TRAIN_EPOCHS = 150
TRAIN_BATCH_SIZE = 8
TRAIN_NUM_POINTS = 1024
TRAIN_WORKERS = 0
TRAIN_LR = 1e-3
TRAIN_WEIGHT_DECAY = 1e-4
TRAIN_DROPOUT = 0.4
TRAIN_LABEL_SMOOTHING = 0.1
TRAIN_FEATURE_TRANSFORM_WEIGHT = 1e-3
TRAIN_USE_CLASS_WEIGHTS = False
TRAIN_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TRAIN_SEED = 0

set_seed(TRAIN_SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_dataset, test_dataset, train_loader, test_loader = get_modelnet40_dataloaders(
    root=MODELNET_ROOT,
    batch_size=TRAIN_BATCH_SIZE,
    num_points=TRAIN_NUM_POINTS,
    workers=TRAIN_WORKERS,
    seed=TRAIN_SEED,
    sample_method='surface',
)

labels = train_dataset.labels
labels_path = OUTPUT_DIR / 'labels.txt'
with open(labels_path, 'w', encoding='utf-8') as handle:
    for label in labels:
        handle.write(f'{label}\n')

class_weights = None
if TRAIN_USE_CLASS_WEIGHTS:
    class_weights = compute_class_weights(train_dataset, len(labels)).to(TRAIN_DEVICE)

model = PointNetCls(k=len(labels), dropout=TRAIN_DROPOUT).to(TRAIN_DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=TRAIN_LR, weight_decay=TRAIN_WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=TRAIN_EPOCHS,
    eta_min=max(TRAIN_LR * 0.01, 1e-5),
)

best_acc = 0.0
best_ckpt_path = OUTPUT_DIR / 'pointnet_best.pth'
latest_ckpt_path = OUTPUT_DIR / 'pointnet_last.pth'
history = []

print('device =', TRAIN_DEVICE)
print('num_classes =', len(labels))
print('train_size =', len(train_dataset), 'test_size =', len(test_dataset))

In [ ]:
for epoch in range(1, TRAIN_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_seen = 0

    for points, labels_batch in train_loader:
        points = points.to(TRAIN_DEVICE)
        labels_batch = labels_batch.to(TRAIN_DEVICE)

        optimizer.zero_grad()
        logits, trans_feat = model(points)
        loss = F.cross_entropy(
            logits,
            labels_batch,
            weight=class_weights,
            label_smoothing=TRAIN_LABEL_SMOOTHING,
        )
        loss = loss + TRAIN_FEATURE_TRANSFORM_WEIGHT * feature_transform_regularizer(trans_feat)
        loss.backward()
        optimizer.step()

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * labels_batch.size(0)
        total_correct += (preds == labels_batch).sum().item()
        total_seen += labels_batch.size(0)

    train_loss = total_loss / max(total_seen, 1)
    train_acc = total_correct / max(total_seen, 1)
    val_loss, val_acc = evaluate(
        model,
        test_loader,
        TRAIN_DEVICE,
        class_weights=class_weights,
        label_smoothing=TRAIN_LABEL_SMOOTHING,
        feature_transform_weight=TRAIN_FEATURE_TRANSFORM_WEIGHT,
    )
    scheduler.step()

    history.append(
        {
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_loss,
            'val_acc': val_acc,
        }
    )

    print(
        f'epoch {epoch:03d} | train_loss {train_loss:.4f} | '
        f'train_acc {train_acc:.4f} | val_loss {val_loss:.4f} | val_acc {val_acc:.4f}',
        flush=True,
    )

    ckpt = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'labels': labels,
        'num_points': TRAIN_NUM_POINTS,
        'dropout': TRAIN_DROPOUT,
        'label_smoothing': TRAIN_LABEL_SMOOTHING,
        'feature_transform_weight': TRAIN_FEATURE_TRANSFORM_WEIGHT,
        'val_acc': val_acc,
    }
    torch.save(ckpt, latest_ckpt_path)
    if val_acc >= best_acc:
        best_acc = val_acc
        torch.save(ckpt, best_ckpt_path)

metrics_path = OUTPUT_DIR / 'train_metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as handle:
    json.dump(history, handle, indent=2)

print('best checkpoint:', best_ckpt_path)
print('labels file:', labels_path)
print('metrics:', metrics_path)

## 9. 在 notebook 内直接做单样本预测

这里自动从数据集中找一个真实存在的 `.off` 文件，不再写死文件名。

In [ ]:
import numpy as np
import torch

from TSDF.dataset.modelnet40_data import load_off_points
from TSDF.detection.pointnet_model import PointNetCls
from TSDF.detection.train_pointnet_cls import normalize_points

sample_path = sorted(MODELNET_ROOT.rglob('*.off'))[0]
print('using sample:', sample_path)

with open(OUTPUT_DIR / 'labels.txt', 'r', encoding='utf-8') as handle:
    pred_labels = [line.strip() for line in handle if line.strip()]

ckpt = torch.load(OUTPUT_DIR / 'pointnet_best.pth', map_location=TRAIN_DEVICE)
pred_model = PointNetCls(k=len(pred_labels), dropout=ckpt.get('dropout', 0.4)).to(TRAIN_DEVICE)
pred_model.load_state_dict(ckpt['model_state_dict'])
pred_model.eval()

raw_points = load_off_points(sample_path, num_points=TRAIN_NUM_POINTS, rng=np.random.default_rng(0), sample_method='surface')
prepared = normalize_points(raw_points.astype(np.float32))
tensor = torch.from_numpy(prepared.T).unsqueeze(0).to(TRAIN_DEVICE)

with torch.no_grad():
    logits, _ = pred_model(tensor)
    probs = torch.softmax(logits, dim=1)[0]
    pred_idx = int(torch.argmax(probs).item())

topk = min(5, len(pred_labels))
top_probs, top_indices = torch.topk(probs, k=topk)
print('predicted:', pred_labels[pred_idx])
print('confidence:', float(probs[pred_idx]))
print('top_predictions:')
for rank in range(topk):
    cls_idx = int(top_indices[rank].item())
    score = float(top_probs[rank].item())
    print(f'  {rank + 1}. {pred_labels[cls_idx]} ({score:.4f})')

## 10. 保存训练结果回 Drive

In [ ]:
import shutil

DRIVE_EXPORT_DIR.parent.mkdir(parents=True, exist_ok=True)
if DRIVE_EXPORT_DIR.exists():
    shutil.rmtree(DRIVE_EXPORT_DIR)
shutil.copytree(OUTPUT_DIR, DRIVE_EXPORT_DIR)
print('Saved to', DRIVE_EXPORT_DIR)
!ls -lah {DRIVE_EXPORT_DIR}

## 11. 本地可视化

训练完成后，把 Drive 中导出的以下文件下载到本地：

- `pointnet_best.pth`
- `labels.txt`
- 一个 `.off` 文件或你自己的点云

然后在本地仓库中运行：

```bash
python detection/predict_pointnet_pointcloud.py \
  --input path/to/sample.off \
  --checkpoint model/pointnet_modelnet40_run/pointnet_best.pth \
  --labels model/pointnet_modelnet40_run/labels.txt \
  --visualize
```